# 00 — Data Transform

Loads raw AG News CSVs, builds the `text` column, 0-indexes labels, and
creates the semi-supervised labeled/unlabeled splits. Run this first —
every other notebook reads from `data/processed/`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from utils import config
from utils.data import load_raw, build_text_column, make_splits

In [2]:
train_raw = load_raw(config.DATA_DIR / "train.csv")
test_raw = load_raw(config.DATA_DIR / "test.csv")

train_clean = build_text_column(train_raw)
test_clean = build_text_column(test_raw)

train_clean.head()

,label,title,description,text
0,2,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...
1,2,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...
2,2,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...
3,2,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...
4,2,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...","Oil prices soar to all-time record, posing new..."


In [3]:
assert len(train_clean) == 120_000, f"expected 120000 train rows, got {len(train_clean)}"
assert len(test_clean) == 7_600, f"expected 7600 test rows, got {len(test_clean)}"
assert train_clean["text"].isna().sum() == 0
assert test_clean["text"].isna().sum() == 0

train_counts = train_clean["label"].value_counts().sort_index()
test_counts = test_clean["label"].value_counts().sort_index()
assert (train_counts == 30_000).all(), train_counts
assert (test_counts == 1_900).all(), test_counts

print("Train class distribution:\n", train_counts)
print("\nTest class distribution:\n", test_counts)
print("\nSample rows:\n", train_clean.sample(3, random_state=config.SEED))

Train class distribution:
 label
0    30000
1    30000
2    30000
3    30000
Name: count, dtype: int64

Test class distribution:
 label
0    1900
1    1900
2    1900
3    1900
Name: count, dtype: int64

Sample rows:
        label                                         title  \
71787      2  BBC set for major shake-up, claims newspaper   
67218      2                      Marsh averts cash crunch   
54066      1      Jeter, Yankees Look to Take Control (AP)   

                                             description  \
71787  London - The British Broadcasting Corporation,...   
67218  Embattled insurance broker #39;s banks agree t...   
54066  AP - Derek Jeter turned a season that started ...   

                                                    text  
71787  BBC set for major shake-up, claims newspaper L...  
67218  Marsh averts cash crunch Embattled insurance b...  
54066  Jeter, Yankees Look to Take Control (AP) AP - ...  


In [4]:
labeled_df, unlabeled_df = make_splits(
    train_clean, label_fraction=config.LABEL_FRACTION, seed=config.SEED)

assert len(labeled_df) + len(unlabeled_df) == len(train_clean)
assert (unlabeled_df["label"] == -1).all()
print(f"Labeled pool: {len(labeled_df)} rows ({config.LABEL_FRACTION:.0%})")
print(f"Unlabeled pool: {len(unlabeled_df)} rows (true_label hidden for eval only)")

Labeled pool: 6000 rows (5%)
Unlabeled pool: 114000 rows (true_label hidden for eval only)


In [5]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train_clean.to_parquet(config.PROCESSED_DIR / "train_clean.parquet", index=False)
test_clean.to_parquet(config.PROCESSED_DIR / "test_clean.parquet", index=False)
labeled_df.to_parquet(config.PROCESSED_DIR / "labeled.parquet", index=False)
unlabeled_df.to_parquet(config.PROCESSED_DIR / "unlabeled.parquet", index=False)
print("Saved processed splits to", config.PROCESSED_DIR)

Saved processed splits to C:\Users\ACER\OneDrive\Documents\final-project\.worktrees\autolabel-notebooks\data\processed
